# Exercise 21.2: A complex cardiac calcium model

In this exercise, you will wire together the massive **Jafri-Winslow-Borg (1998)** cardiomyocyte model. This model differentiates between 4 spatial calcium compartments ($Ca_{SS}$, $Ca_{i}$, $Ca_{JSR}$, $Ca_{NSR}$) and contains 31 state variables!

Because implementing 31 ODEs by hand guarantees typos, the core logic has been isolated into an external file (`Jafri_model.py`). Your job is to fill in the missing complex formulations (the Luo-Rudy NCX, the RyR Markov states, and the compartment balance equations).


## Exercise 21.2a: Completing the RHS

Complete the missing blocks in the `rhs` function below. Refer back to Chapter 20 for the Luo-Rudy equation, and carefully track the signs of the compartment fluxes!


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from Jafri_model import Jafri_model_parts


def rhs_jafri(t, y):
    # Split up the 31-state vector
    (
        V,
        Nai,
        m,
        h,
        j,
        O,
        O_Ca,
        C0,
        C1,
        C2,
        C3,
        C4,
        C_Ca0,
        C_Ca1,
        C_Ca2,
        C_Ca3,
        C_Ca4,
        Ca_SS,
        Ko,
        Ki,
        y_gate,
        X,
        Cai,
        P_O1,
        P_O2,
        P_C1,
        P_C2,
        Ca_JSR,
        Ca_NSR,
        HTRPNCa,
        LTRPNCa,
    ) = y

    # Constants & Params (truncated for brevity, assume they are defined here as in the original code block)
    R, T, F, Cm = 8.3145e3, 310, 9.6845e4, 0.01
    Nao, Cao = 140, 1.8
    k_NaCa, K_mNa, K_mCa, k_sat, eta = 50, 87.5, 1.38, 0.1, 0.35
    v1, v2, v3 = 1.8, 0.58e-4, 1.8e-3
    k_a_plus, k_a_minus = 1.215e10, 0.1425
    k_b_plus, k_b_minus = 4.05e7, 1.93
    k_c_plus, k_c_minus = 0.018, 0.0008
    tau_tr, tau_xfer = 34.48, 3.125
    K_mup, K_mCMDN, K_mCSQN = 0.5e-3, 2.38e-3, 0.8
    CSQN_tot, CMDN_tot = 15, 0.05
    Am, V_myo = 546.69, 0.92
    V_SS, V_NSR, V_JSR = 5.828e-05 * V_myo, 0.081 * V_myo, 0.00464 * V_myo
    conv_Amp_myo = Am / (2.0 * V_myo * F)

    # 1. Fetch pre-computed currents from the external file
    (
        dm_dt,
        dh_dt,
        dj_dt,
        i_Na,
        dX_dt,
        i_K,
        i_K1,
        i_Kp,
        i_NaK,
        i_ns_Ca,
        i_ns_Na,
        i_ns_K,
        i_p_Ca,
        i_Ca_b,
        i_Na_b,
        dy_dt,
        dC0_dt,
        dC1_dt,
        dC2_dt,
        dC3_dt,
        dC4_dt,
        dC_Ca0_dt,
        dC_Ca1_dt,
        dC_Ca2_dt,
        dC_Ca3_dt,
        dC_Ca4_dt,
        dO_dt,
        dO_Ca_dt,
        dHTRPNCa_dt,
        dLTRPNCa_dt,
        i_Ca_L_Ca,
        i_Ca_L_K,
        J_trpn,
    ) = Jafri_model_parts().currents_concentrations(
        V,
        m,
        h,
        j,
        Nai,
        X,
        Ko,
        Ki,
        Cai,
        y_gate,
        C0,
        C1,
        C2,
        C3,
        C4,
        C_Ca0,
        C_Ca1,
        C_Ca2,
        C_Ca3,
        C_Ca4,
        O,
        O_Ca,
        Ca_SS,
        Ca_JSR,
        Ca_NSR,
        HTRPNCa,
        LTRPNCa,
    )

    # 2. Implement the Luo-Rudy NCX formulation
    num = (Nai**3 * Cao) * math.exp(eta * V * F / (R * T)) - (Nao**3 * Cai) * math.exp(
        (eta - 1) * V * F / (R * T)
    )
    den = (
        (K_mNa**3 + Nao**3)
        * (K_mCa + Cao)
        * (1 + k_sat * math.exp((eta - 1) * V * F / (R * T)))
    )
    i_NaCa = k_NaCa * num / den

    # 3. Implement the RyR 4-state Markov Model
    RyR_open = P_O1 + P_O2
    J_rel = v1 * RyR_open * (Ca_JSR - Ca_SS)

    dP_C1_dt = -k_a_plus * (Ca_SS**nCa) * P_C1 + k_a_minus * P_O1
    dP_O1_dt = (
        k_a_plus * (Ca_SS**nCa) * P_C1
        - k_a_minus * P_O1
        - k_b_plus * (Ca_SS**mCa) * P_O1
        + k_b_minus * P_O2
        - k_c_plus * P_O1
        + k_c_minus * P_C2
    )
    dP_O2_dt = k_b_plus * (Ca_SS**mCa) * P_O1 - k_b_minus * P_O2
    dP_C2_dt = k_c_plus * P_O1 - k_c_minus * P_C2

    # 4. Calcium subsystem fluxes
    J_leak = v2 * (Ca_NSR - Cai)
    J_up = (v3 * (Cai**2.0)) / ((K_mup**2.0) + (Cai**2.0))
    J_tr = (Ca_NSR - Ca_JSR) / tau_tr
    J_xfer = (Ca_SS - Cai) / tau_xfer

    # 5. Compartment differentials (incorporating buffers Bi, B_JSR, B_SS)
    Bi = 1.0 / (1.0 + (CMDN_tot * K_mCMDN) / ((K_mCMDN + Cai) ** 2.0))
    B_JSR = 1.0 / (1.0 + (CSQN_tot * K_mCSQN) / ((K_mCSQN + Ca_JSR) ** 2.0))
    B_SS = 1.0 / (1.0 + (CMDN_tot * K_mCMDN) / ((K_mCMDN + Ca_SS) ** 2.0))

    # Fill in the sum of fluxes entering/leaving each compartment!
    dCa_SS_dt = B_SS * (
        J_rel - J_xfer
    )  # Example: Add L-type calcium current converting Amp to concentration
    dCa_JSR_dt = B_JSR * (J_tr - J_rel)
    dCa_NSR_dt = J_up - J_leak - (J_tr * V_JSR / V_NSR)
    dCai_dt = Bi * (
        J_xfer * V_SS / V_myo - J_up + J_leak - J_trpn
    )  # Plus boundary fluxes

    # 6. Transmembrane potential
    I_stim = 0.516289 if (100 <= t <= 101) else 0.0  # Simplified stimulus
    I_tot = (
        i_Na
        + i_K
        + i_K1
        + i_Kp
        + i_NaK
        + i_NaCa
        + i_p_Ca
        + i_Ca_L_Ca
        + i_Ca_L_K
        + i_ns_Ca
        + i_ns_Na
        + i_ns_K
        + i_Ca_b
        + i_Na_b
    )
    dV_dt = (I_stim - I_tot) / Cm

    # Other basic ion differentials (simplified for brevity)
    dNai_dt = 0
    dKi_dt = 0
    dKo_dt = 0

    return [
        dV_dt,
        dNai_dt,
        dm_dt,
        dh_dt,
        dj_dt,
        dO_dt,
        dO_Ca_dt,
        dC0_dt,
        dC1_dt,
        dC2_dt,
        dC3_dt,
        dC4_dt,
        dC_Ca0_dt,
        dC_Ca1_dt,
        dC_Ca2_dt,
        dC_Ca3_dt,
        dC_Ca4_dt,
        dCa_SS_dt,
        dKo_dt,
        dKi_dt,
        dy_dt,
        dX_dt,
        dCai_dt,
        dP_O1_dt,
        dP_O2_dt,
        dP_C1_dt,
        dP_C2_dt,
        dCa_JSR_dt,
        dCa_NSR_dt,
        dHTRPNCa_dt,
        dLTRPNCa_dt,
    ]
